# Load pytorch model

In [1]:
import subprocess

try:
    cuda_version = subprocess.check_output(["nvcc", "--version"]).decode().split("release ")[-1].split(",")[0]
    print(f"CUDA version: {cuda_version}")
except:
    print("Unable to determine CUDA version. Make sure CUDA is installed and nvcc is in your PATH.")

import os

# import sys
# sys.path.append('/opt/oneflow/python')


CUDA version: 12.2


In [2]:
import os
import time
import requests
from tqdm import tqdm
import numpy as np

import diffusers
from diffusers import AutoencoderKL, AutoencoderTiny, AutoPipelineForImage2Image

# from DeepCache import DeepCacheSDHelper

import oneflow as flow
from onediff.infer_compiler import oneflow_compile

# import xformers
# import triton
# from sfast.compilers.diffusion_pipeline_compiler import (compile, CompilationConfig)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
import oneflow as flow

print(f"PyTorch version: {torch.__version__}")
print(f"OneFlow version: {flow.__version__}")
print(f"cudnn version: {torch.backends.cudnn.version()}")
print(f"CUDA version: {torch.version.cuda}")

if not torch.cuda.is_available():
    raise SystemError("GPU device not found. PyTorch requires a GPU to run this code.")
else:
    print(f"Found GPU device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")

PyTorch version: 2.1.0a0+32f93b1
OneFlow version: 0.9.1.dev20240515+cu122
cudnn version: 8907
CUDA version: 12.2
Found GPU device: NVIDIA GeForce RTX 4090
CUDA capability: (8, 9)


In [4]:
def download_checkpoint(download_url, checkpoint_file):
    if os.path.exists(checkpoint_file):
        print(f"{checkpoint_file} already exists. Skipping download.")
        return

    folder, filename = os.path.split(checkpoint_file)
    os.makedirs(folder, exist_ok=True)
    with requests.get(download_url, stream=True) as response:
        total_size = int(response.headers.get("content-length", 0))
        with open(checkpoint_file, "wb") as file_stream, tqdm(
            desc=checkpoint_file,
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as progress_bar:
            for data in response.iter_content(chunk_size=1024):
                file_stream.write(data)
                progress_bar.update(len(data))

    print("Download completed successfully.")
download_checkpoint("https://civitai.com/api/download/models/173961", "checkpoints/AnimeV3.safetensors")

checkpoints/AnimeV3.safetensors already exists. Skipping download.


In [5]:
from diffusers import (
    DPMSolverMultistepScheduler,
    EulerDiscreteScheduler,
    EulerAncestralDiscreteScheduler
)

def set_scheduler(scheduler_name: str, config):
    if scheduler_name == "dpm++2m_karras":
        scheduler = DPMSolverMultistepScheduler.from_config(
            config, use_karras_sigmas=True
        )
    elif scheduler_name == "euler":
        scheduler = EulerDiscreteScheduler.from_config(
            config
        )
    elif scheduler_name == "euler_ancestral":
        scheduler = EulerAncestralDiscreteScheduler.from_config(
            config
        )
    else:
        raise ValueError(f"Unknown scheduler: {scheduler_name}")

    return scheduler

# vae = AutoencoderKL.from_pretrained(
#   'madebyollin/sdxl-vae-fp16-fix',
#   use_safetensors=True,
#   torch_dtype=torch.float16,
# ).to('cuda')

vae = AutoencoderTiny.from_pretrained(
  'madebyollin/taesdxl',
  use_safetensors=True,
  torch_dtype=torch.float16,
).to('cuda')

txt2img_pipe = diffusers.StableDiffusionXLPipeline.from_single_file(
    "checkpoints/AnimeV3.safetensors",
    use_safetensors=True,
    vae=vae,
    torch_dtype=torch.float16 #if mode == 16 else torch.float32
)
scheduler_name = "dpm++2m_karras"
# scheduler_name = "euler"
# scheduler_name = "euler_ancestral"
txt2img_pipe.scheduler = set_scheduler(
    scheduler_name, txt2img_pipe.scheduler.config
)
txt2img_pipe.to("cuda")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


StableDiffusionXLPipeline {
  "_class_name": "StableDiffusionXLPipeline",
  "_diffusers_version": "0.26.3",
  "feature_extractor": [
    null,
    null
  ],
  "force_zeros_for_empty_prompt": true,
  "image_encoder": [
    null,
    null
  ],
  "scheduler": [
    "diffusers",
    "DPMSolverMultistepScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "text_encoder_2": [
    "transformers",
    "CLIPTextModelWithProjection"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "tokenizer_2": [
    "transformers",
    "CLIPTokenizer"
  ],
  "unet": [
    "diffusers",
    "UNet2DConditionModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderTiny"
  ]
}

In [6]:
prompts = [
    "Portrait shot of a woman, yellow shirt, photograph",
    "Little girl holding a teddy bear, in the middle of nowhere, photograph",
    "Portrait of an arctic fox in the tundra, light teal and amber, minimalist, photograph",
    "Confused woman, sci - fi, future, blue glow color, orange, hologram, photograph",
    "Symmetrical, macro shot, crying womans face, half of face is organic flowing RGB low poly, depth of field",
    "Beautiful woman future funk psychedelic",
    "Mosaic of a colorful mushroom with intricate patterns, vibrant and detailed, sharp, mosaic background, vector art",
    "Illustration of a man in red hoodie, minimalist, graphic design poster art, dark cyan and sky - blue, honeycore",
    "a bottle of perfume on a clean backdrop, surrounded by fragrant white flowers, product photography, minimalistic, natural light",
    "a bedroom with large windows and modern furniture, gray and gold, luxurious, mid century modern style",
    "an aerial drone shot of the breathtaking landscape of the Bora Bora islands, with sparkling waters under the sun",
    "extreme closeup shot of an old man with a long gray hair and head covered in wrinkles; focused expression looking at camera",
    "Simple flat vector illustration of a woman sitting at the desk with her laptop with a puppy, isolated on white background",
    "Chibi pixel art, game asset for an rpg game on a white background featuring the armor of a dragon sorcerer wielding the power of fire surrounded by a matching item set",
    "a macro wildlife photo of a green frog in a rainforest pond, highly detailed, eye-level shot",
    "kid's coloring book, a happy young girl holding a flower, cartoon, thick lines, black and white, white background",
    "Golden-haired elementary school white boy hugging his black-hair Taiwanese buddy face-to-face on dusk street, unreal engine, greg rutkowski, loish, rhads, beeple, makoto shinkai and lois van baarle, ilya kuvshinov, rossdraws, tom bagshaw, alphonse mucha, global illumination, detailed and intricate environment",
    "Tan skin Anime boy wearing a large black sweater and cat ear beanie with brown hair and eyes, full body, baggy cargo pants, full body, reference",
    "Fawn French Bulldog with big eyes, short legs, and chunky, stocky body eating food",
    "A white goose holding a paint brush",
    "Black, African descent, looks Japanese, wears glasses, Naruto type art, bandage on his nose, male, Anime 2D art, lazy eyes, Japanese earring in one ear, no beard, smiles sinisterly",
    "Male cow fursona wearing a red beanie",
    "a beautiful hyper-realistic anime Lofi, painted by greg rutkowski makoto shinkai takashi takeuchi studio ghibli, akihiko yoshida, anime, clean soft lighting, finely detailed features, high-resolution, perfect art, stunning atmosphere, trending on pixiv fanbox",
    "a woman with a beautiful face is enjoying a summer festival wearing a kimono, long white hair, looks like an older sister with a small body, is holding a traditional Japanese umbrella with a faint smile, her head is facing backwards as if inviting her to play and she is running with her arms behind her, there is also a lock of patterned hair flower",
    "A stunning photograph of a serene mountain lake at sunrise, with crystal-clear reflections and soft pastel skies",
    "A high-resolution image of an ancient oak tree in a lush forest, sunlight filtering through the leaves",
    "An ultra-realistic photograph of the Milky Way galaxy seen from a remote desert, under clear skies",
    "A detailed image of a colorful street market in Marrakech at golden hour, with vibrant fabrics and bustling crowds",
    "A professional photograph of a majestic bald eagle in flight, with a crisp focus on its sharp eyes and detailed feathers",
    "A perfect image of a charming cobblestone street in Prague, with historical buildings and a peaceful early morning atmosphere",
    "A photo-realistic image of a modern city skyline at night, with shimmering lights and reflections on a river",
    "An authentic-looking photograph of the Northern Lights over a snowy Lapland landscape, with vivid colors and clear stars",
    "A high-quality image of a vintage 1950s diner, with classic cars parked outside and a sunset backdrop",
    "An elegant photograph of a grand ballroom from the Victorian era, with ornate decorations and a grand chandelier",
    "A striking photograph of a powerful thunderstorm over the ocean, with dramatic lightning strikes and rolling waves",
    "An image of a peaceful Zen garden with smooth stones, raked sand, and a calming waterfall",
    "A high-resolution photograph of a seasoned fisherman at dawn, casting a net into the sea, with the golden light reflecting off the water",
    "A professional close-up shot of a woman's face, half-illuminated by the sunset, showcasing a detailed texture of her skin and a contemplative expression",
    "An image capturing a street dancer in mid-air during a dynamic breakdance move, with urban graffiti in the background",
    "A vibrant photograph of a group of people dressed in traditional attire at a cultural festival, dancing in a blur of colors and fabrics",
    "A cinematic-style photograph of a lone astronaut in a spacesuit, standing on a rocky alien landscape with Earth visible in the sky above",
    "Capture the quiet intensity in the eyes of a chess grandmaster poised over the board in a high-stakes match",
    "Close-up: A young girl's freckled face, focused and thoughtful, as she reads a book under the shade of an old tree",
    "Underwater photography of a diver among swirling schools of fish, light filtering down from above",
    "Evening falls on a city street musician, his guitar casting long shadows as he strums for the passing crowd",
    "High above the city, a construction worker perches on a steel beam, with a backdrop of the skyline stretching into the distance",
    "Document the intense expression of a potter as they shape a clay vessel, hands and wheel both a blur of motion",
    "A street portrait captures the weathered face of a long-time vendor, his cart a staple in the neighborhood for generations",
    "During golden hour, a group of children race through a field, their silhouettes a dance of joy against the setting sun",
    "Zoomed-in shot capturing the intense focus of a violinist as the bow gracefully sweeps across the strings, emotions etched into their performance",
]
negative_prompt = "out of frame, nude, duplicate, watermark, signature, mutated, text, blurry, worst quality, low quality, artificial, texture artifacts, jpeg artifacts"


In [7]:
import PIL
import os

# Create folder tests if needed
if not os.path.exists("tests"):
    os.makedirs("tests")

In [8]:
import numpy as np

torch_compile = False
backends_cudnn_allow_tf32 = False
deep_cache = False
use_oneflow = True
stable_fast = False

if backends_cudnn_allow_tf32:
    torch.backends.cudnn.allow_tf32 = False

if deep_cache:
    print("Enabling DeepCache")
    helper = DeepCacheSDHelper(txt2img_pipe)
    helper.set_params(cache_interval=3, cache_branch_id=0)
    helper.enable()

if torch_compile:
    txt2img_pipe.unet = torch.compile(txt2img_pipe.unet, mode='max-autotune', fullgraph=True)

if use_oneflow:
    txt2img_pipe.unet = oneflow_compile(txt2img_pipe.unet)

def callback_dynamic_cfg(pipe, step_index, timestep, callback_kwargs):
  if step_index == int(pipe.num_timesteps * 0.4):
    callback_kwargs['prompt_embeds'] = callback_kwargs['prompt_embeds'].chunk(2)[-1]
    callback_kwargs['add_text_embeds'] = callback_kwargs['add_text_embeds'].chunk(2)[-1]
    callback_kwargs['add_time_ids'] = callback_kwargs['add_time_ids'].chunk(2)[-1]
    pipe._guidance_scale = 0.0

  return callback_kwargs

# refiner = AutoPipelineForImage2Image.from_pretrained(
#   'stabilityai/stable-diffusion-xl-refiner-1.0',
#   use_safetensors=True,
#   torch_dtype=torch.float16,
#   variant='fp16',
# ).to('cuda')

# if stable_fast:
#   config = CompilationConfig.Default()

#   config.enable_xformers = True
#   config.enable_triton = True
#   config.enable_cuda_graph = True

#   txt2img_pipe = compile(txt2img_pipe, config)



WARNING [2024-07-08 22:22:20] /usr/local/lib/python3.10/dist-packages/onediff/infer_compiler/backends/oneflow/transform/manager.py:123 - Pydantic version 2.4.2 is too low, please upgrade to 2.5.2 or higher.


In [9]:
# with flow.autocast('cuda'):
    # torch.manual_seed(42)
result = []
for i, prompt in enumerate(prompts):
    torch.manual_seed(42)
    t1 = time.time()
    image = txt2img_pipe(prompt=prompt, negative_prompt=negative_prompt, num_inference_steps=20, 
                        callback_on_step_end=callback_dynamic_cfg,
    callback_on_step_end_tensor_inputs=['prompt_embeds', 'add_text_embeds', 'add_time_ids'],
                        guidance_scale=7, width=1024, height=1024, 
                        num_images_per_prompt = 1).images[0]
    t2 = time.time() - t1
    result.append(t2)
    # image.save(f"tests/{i}-original.jpg")
    # image.save(f"tests/{i}-test.jpg")
np.mean(result), min(result), max(result)

100%|██████████| 20/20 [00:00<00:00, 20.32it/s]


(1.7551883220672608, 1.0510196685791016, 35.73342442512512)

In [10]:
np.mean(result[1:]), result

(1.061754932208937,
 [35.73342442512512,
  1.0510196685791016,
  1.0522782802581787,
  1.0536043643951416,
  1.054783821105957,
  1.0537960529327393,
  1.055595874786377,
  1.0550661087036133,
  1.0561141967773438,
  1.0557796955108643,
  1.05698561668396,
  1.0575594902038574,
  1.058103084564209,
  1.0606842041015625,
  1.0576457977294922,
  1.0594615936279297,
  1.0626378059387207,
  1.060441255569458,
  1.0599722862243652,
  1.059523105621338,
  1.061786413192749,
  1.0599100589752197,
  1.0634605884552002,
  1.0646896362304688,
  1.06113862991333,
  1.0612881183624268,
  1.0616471767425537,
  1.062502145767212,
  1.0633349418640137,
  1.0635581016540527,
  1.0628447532653809,
  1.0635101795196533,
  1.0638861656188965,
  1.0649614334106445,
  1.064466953277588,
  1.064321517944336,
  1.0654172897338867,
  1.0656840801239014,
  1.0666446685791016,
  1.0667614936828613,
  1.0658643245697021,
  1.0662267208099365,
  1.0670998096466064,
  1.067009449005127,
  1.067739725112915,
  1.06